In [1]:
import torch


# Modelos de secuencias

Hasta ahora hemos trabajado con series de datos donde a cada entrada le corresponde una salida. Por ejemplo, a una imagen le corresponde una categoría. A una serie de indicadores biométricos le corresponde un diagnóstico médico.

En procesamiento de lenguajes naturales, nuestras salidas y nuestras entradas tienen una característica distinta. Veamos un ejemplo:

> *Usted tiene 16 años. Está prohibido vender alcohol a menores de 18 años. No puedo venderle esa botella.*

En el ejemplo anterior, tenemos tres afirmaciones, donde la última es un conclusión de las dos anteriores. En este sentido, cuando trabajamos con lenguajes tenemos el problema de lo próximo que se dice, depende de lo que se dijo antes. Es decir, estamos trabajando con secuencias temporales.

Peor aún, muchas veces la última salida, debe ser tratada como una nueva entrada. Piense en el ejemplo anterior, si usted vive en un país latinoamericano o europeo, al llegar a **menores de** intuía que la edad límite sería **18 años**. Eso es porque como ciudadano de su país, sabe que esa es la ley. Es decir. **18 años** podría haber sido una predicción, una salida de su red. Ademas, al haber predicho **18 años** ahora podemos concluir que **No puedo venderle esa botella**. Si la ley dijera que los menores de **14 años** pueden comprar alcohol, la segunda frase carecería de sentido. Es decir **18 años** es una predicción, una salida en un momento, pero luego se convierte en una entrada o un *feature* en otro.

Es por lo anterior que se dice que estos modelos son modelos autoregresivos: las salidas luego se convierten en entradas, como en un problema recursivo.

La naturaleza autogresiva de nuestro modelo hace que debamos considerar la calidad de nuestras predicciones. Volviendo al ejemplo anterior, si nuestra predicción hubiera sido **14 años** en lugar de **18 años**, la conclusión final de nuestra frase sería distinta a la que hemos obtenido. Pequeños errores en un nuestras predicciones pueden acumularse a lo largo del tiempo y generar resultados absurdos.



# Modelos markovianos y variables ocultas.

Volvamos de nuevo a nuestro ejemplo de oraciones

> *Usted tiene 16 años. Está prohibido vender alcohol a menores de 18 años. No puedo venderle esa botella.*

Supongamos de nuevo que queremos predecir **18 años**. La cantidad de palabras escritas hasta ese momento es 11. Luego de predecir **18 años**, la cantidad de palabras aumentó a 13. Es decir, conforme predecimos y agregamos información, nuestro modelo debe responder a la cantidad creciente de palabras o ejemplos.

Recordemos que todas estas herramientas nacieron de la estadística, por lo que nuestras predicciones se basaran en considerar la probabilidad de que diferentes palabras ocurran en simultaneo. Esto es verdaderamente un problema: mientas más palabras tenemos, menos probable es que vuelvan a ocurrir. Si ocurren infrecuentemente, necesitamos aumentar cada vez más la cantidad de ejemplos de nuestros datos. Esto puede ser un problema incluso para oraciones cortas. Una alternativa para paliar este problema es limitar la cantidad de palabra que miraremos hacia atras.

Otra alternativa a esto es trabajar con variables ocultas. Las variables ocultas son cantidades que de alguna manera agrupan la información de todos los casos anteriores. Por ejemplo:

> *Usted es menor de edad. No puedo venderle esa botella.*

Hemos resumido toda la información de dos oraciones en una sola mucho más corta.

De la misma manera que buscamos representaciones abstractas para palabras por medio de *tokens*, usaremos esos tokens para generar nuevas variables que resuman la información anterior. Es decir, generaremos una variable que de alguna manera tiene toda la información de **Usted es menor de edad**

Al trabajar con variables ocultas, esperamos reemplazar todas las palabras anteriroes con el último valor de la variable oculta. Así, nuestro problema que antes veía 13 variables o tokens ahora ve uno solo. Esto nos permite simplificar nuestro modelo para trabajar con **modelos makovianos de primer orden**.

Sin entrar en mucho detalle, los modelos markovianos son el tipo de modelos autoregresivos como los que hemos descripto hasta ahora.

Esbozo de la noción de modelo de Markov

* Tenemos un estado (las últimas palabras escritas) al cual llegamos a partir de un estado inicial bien definido
* Tenemos una historia de estados pasados que afecta a estados futuros (cada palabra predicha o dentro de nuestro dataset)
* Hay una probabilidad asociada a cada cambio de estado
* Queremos predecir cuál será el próximo estado de un grupo finito de estados (la próxima palabra).

Al trabajar con un modelo markoviano sobre variables ocultas, esperamos que la variable oculta resuma con tanta fidelidad los tokens pasados que solo necesitemos la variable oculta más reciente. Al necesitar solo la más reciente, se dice que es un modelo markoviano de primer orden (requiera solo una variable anterior). La razón por la que buscamos trabajar con modelos de primer orden es que son menos costosos computacionalmente.

En resumen nuestra propuesta para generar modelos de lenguaje consistirá en lo siguiente:

1. Tomaremos texto para crear nuestro *dataset*
1. Transformaremos nuestro texto en algun tipo de representación simbólica (*tokens*)
2. De esta manera, nuestro modelo de lenguaje se convertirá en un problema de clasificación: Dadas las palabras anteriores, ¿Cuál es la siguiente palabra?
  * Decimos que es un problema de clasificación, porque cada una de nuestra palabras es una categoría.
3. Para crear nuestro modelo de lenguaje, usaremos variables ocultas en el contexto de un modelo markoviano.
  * La justificación para ésto es que el lenguaje tiene características de un modelo markoviano.



# Redes neuronales recurrentes

En la sección anterior intentamos argumentar que el lenguaje puede modelarse con un modelo markoviano. Además, propusimos trabajar con modelos markovianos con variables ocultas. La idea de trabajar con variables ocultas es poder trabajar con un modelo markoviano de primer orden. Queremos usar estos modelos de primer orden porque sabemos que nos permitirán ahorrar uso de memoria, así como disminuir el uso de recursos computacionales.

Nuestra propuesta para trabajar con variables ocultas, será trabajar con las unidades ocultas de un perceptrón multicapa

![](http://d2l.ai/_images/mlp.svg)

$$\mathbf{O} = \mathbf{H} \mathbf{W} + \mathbf{b}$$
$$\mathbf{H} = \phi(\mathbf{X} \mathbf{W} + \mathbf{b})$$

Sin embargo, como trabajaremos con secuencias temporales, nuestra entrada al tiempo $t$, debe depender del tiempo $t-1$. Es decir, la próxima palabra debe depender de las palabras anteriores. En un perceptrón multicapa, esa dependecia temporal no está presente. Es por esto que debemos reestructurar nuestra capa para que permita generar modelos autoregresivos de secuencias. Dado que queremos usar las variables ocultas como cantidades que resumen toda la información anterior, son estas cantidades las que tendrán una dependencia temporal

$$\mathbf{O}_{t} = \mathbf{H}_{t} \mathbf{W}_{O} + \mathbf{b}$$
$$\mathbf{H}_t = \phi(\mathbf{X}_t \mathbf{W}_{X} + \mathbf{H}_{t-1} \mathbf{W}_{H}  + \mathbf{b}_h).$$

Notemos que $\mathbf{H}_t$ depende del valor anterior, $\mathbf{H}_{t-1}$ y del nuevo valor $\mathbf{X}_t$. Esta dependencia temporal es la que hace que nuestra nueva red neuronal sea una *red neuronal recurrente*. Recordemos que si nuestro modelo está correctamente entrenado, la salida $\mathbf{O}_t$ debe coincidir con el resultado correcto o *grounding truth* de $\mathbf{X}_{t+1}$. Esta era la naturaleza autoregresiva de nuestros modelos.

En la siguiente figura mostramos el proceso de cálculo nuestra capa recurrente

![An RNN with a hidden state.](http://d2l.ai/_images/rnn.svg)

En la figura, vemos qué ocurre a cada instante $t$:

1. Tenemos una capa densa con función de activación $\phi$ que toma nuestra matriz de diseño $\mathbf{X}_t$ y nuestra variable oculta $\mathbf{H}_{t-1}$.
2. A la salida generamos nuestra nueva variable oculta $\mathbf{H}_t$.
3. Con $\mathbf{H}_t$ y otra capa densa generamos nuestra salida $\mathbf{O}_t$

Muchas veces, en el paso 1 lo que se hace es concatenar $\mathbf{X}_t$ y $\mathbf{H}_{t-1}$ para de esa manera definir una única matriz de pesos. A continuación mostramos cómo esta concatenación genera el mismo resultado.

In [ ]:
X, W_xh = torch.randn(3, 1), torch.randn(1, 4)  # Tres vectores de entrada (3x1) y pesos entrada-oculta (1x4)
H, W_hh = torch.randn(3, 4), torch.randn(4, 4)  # Estados ocultos previos para cada secuencia (3x4) y pesos recurrentes (4x4)

torch.matmul(X, W_xh) + torch.matmul(H, W_hh)   # Calcula la entrada neta a la capa oculta: término de entrada + término recurrente


tensor([[  2.1282,   3.1709,  -0.5072,  -6.0984],
        [  1.1084, -13.3446,   2.1054,   2.1267],
        [  2.9530,  -6.4269,   4.0083,  -2.0037]])

In [3]:
torch.matmul(torch.cat((X, H), 1), torch.cat((W_xh, W_hh), 0))

tensor([[  2.1282,   3.1709,  -0.5072,  -6.0984],
        [  1.1084, -13.3446,   2.1054,   2.1267],
        [  2.9530,  -6.4269,   4.0083,  -2.0037]])

## Problemas con nuestras predicciones

Un último punto que hemos evitado hasta ahora es el problema de la predicción en secuencias temporales. Dijimos más arriba que la salida $\mathbf{O}_t$ debe coincidir con el resultado correcto o *grounding truth* de $\mathbf{X}_{t+1}$. Sin embargo, para entrenar correctamente nuestro modelo debemos cada tanto usar la salida $\mathbf{O}_t$ en lugar de $\mathbf{X}_{t+1}$. Si usamos siempre nuestra *grounding truth* en lugar de nuestra predicción, corremos el riesgo de que nuestra red no aprenda a adaptarse a malas predicciones que genera. Al mismo tiempo, nuestra red aprende a modelar las secuencias de entrenamiento, pero puede no saber qué hacer con oraciones nuevas.

El ejemplo sería como el siguiente:

Entrenamos una red con "El ingenioso hidalgo Don Quijote de la Mancha". Luego usamos la red para completar:

> *En un lugar de la Mancha de cuyo nombre prefiero no ...*

La red predice "recordar", en lugar de "acordarme". En el siguiente paso, si usamos "acordarme" la nueva predicción podría ser "vivía", pero si usaramos "recordar" la predicción podría ser "Residía". Es decir, al usar solo la *grounding truth* en lugar de nuestra predicción, el modelo no aprende a corregir sus errores, y se queda "estancado" en los datos de entrenamiento. El mayor problema es que cuando tengamos un modelo funcionando, NO HAY *GROUNDING TRUTH*. Esto hace que nuestro modelo pueda fallar estrepitosamente si no usamos nuestra predicción como entrada al modelo. Pero al mismo tiempo, si usamos siempre nuestra predicción, los errores se acumularán paulatinamente.

Más adelante hablaremos de la técnica llamada **teacher forcing** y cómo elegir cuándo usar la predicción y el *grounding truth* para generar redes que puedan adaptarse a la variabilidad de nuestra predicción.

# Preliminares a la implementación de RNN

Antes de discutir pasar a implementar una RNN desde cero, queremos discutir algunos temás más que serán importantes conocer.



## Muestro de secuencias.

Cuando teníamos que elegir qué ejemplos usar de un dataset que no contiene secuencias, simplemente mezclábamos aleatoriamente los ejemplos y luego los usábamos para entrenar nuestras redes.

Sin embargo, en secuencias temporales no podemos hacer esto. Si mezclamos aleatoriamente podemos terminar generando secuencias sin sentido.

> *En un lugar de la Mancha de cuyo nombre prefiero no acordarme*

luego de mezclarlo

> *nombre En de de no lugar prefiero la Mancha cuyo acordarme un*

Por esta razón debemos generar particiones y mezclarlas. Por ejemplo, podemos generar particiones de 4 elementos

> [*En un lugar de*] [*la Mancha de cuyo*] [*nombre prefiero no acordarme*]

> [*la Mancha de cuyo*][*nombre prefiero no acordarme*] [*En un lugar de*]

Además de eso, podemos elegir un offset o desplazamiento. En el ejemplo anterior, un offset de 1 generaría:

> *En* [*un lugar de la*] [*Mancha de cuyo nombre*] [*prefiero no acordarme, no*]



## Perplejidad

La perplejidad es una métrica que es usada en procesamiento de lenguajes naturales para tener una idea de que tan "convencido" está  nuestro modelo de la siguiente palabra que adivinará. Como métrica está relacionada a la entropía y la entropía cruzada, por lo que usaremos los ejemplos del sgte juego.

Una versión simple de nuestro juego:

* Materiales:
  * una bolsa o recipiente opaco
  * 4 pelotas con los números 1, 2, 3, 4
* Preparativos:
  * Se colocan las pelotas en la bolsa
  * El primer jugador saca una de las pelotas de la bolsa
* Objetivo general: Adivinar con el menor número de preguntas posibles cuál es el número de la pelota que tiene el primer jugador .
  * Solo pueden hacerse preguntas que tengan como respuestas Sí o No.

Tenemos una estrategia óptima:

```
1. Preguntar: "¿El número es par?"
  A. Si la respuesta es sí, preguntar: "¿Es el número 4?"
    a. Si la respuesta es sí, sabemos que es el número 4, hemos ganado.
    b. Si la respuesta es no, sabemos que es el número 2, hemos ganado.
  B. Si la respuesta es no, preguntar: "¿Es el número 3?"
    a. Si la respuesta es sí, sabemos que es el número 3, hemos ganado.
    b. Si la respuesta es no, sabemos que es el número 1, hemos ganado.
```

Recordemos la entropía de nuestra estrategia:

||$1$|$2$|$3$|$4$|total|
|---|---|---|---|---|:-:|
|probabilidad de ocurrir|$\dfrac{1}{4}$|$\dfrac{1}{4}$|$\dfrac{1}{4}$|$\dfrac{1}{4}$|-|
|número de preguntas|$2$|$2$|$2$|$2$|-|
|producto|$\dfrac{1}{2}$|$\dfrac{1}{2}$|$\dfrac{1}{2}$|$\dfrac{1}{2}$|$2$|

Ahora preguntamos, ¿cuántas opciones posibles tenemos en nuestro juego?
> Como todas las pelotas son equiprobables, tenemos 4 opciones distintas.

La pregunta ahora es qué pasará en nuestro segundo juego cuando preguntemos ¿cuántas opciones posibles hay?

Segundo juego:

* Materiales:
  * una bolsa o recipiente opaco
  * 8 pelotas con los números 1, 1, 1, 1, 2, 2, 3, 4
* Preparativos:
  * Se colocan las pelotas en la bolsa
  * El primer jugador saca una de las pelotas de la bolsa
* Objetivo general: Adivinar con el menor número de preguntas posibles cuál el número de la pelota que tiene el primer jugador .
  * Solo pueden hacerse preguntas que tengan como respuestas Sí o No.

Estrategia óptima

```
1. Preguntar: "¿Es el número 1?"
  A. Si la respuesta es sí, hemos ganado."
  B. Si la respuesta es no, preguntar: "¿Es el número 2?"
    a. Si la respuesta es sí, hemos ganado.
    b. Si la respuesta es no, preguntar: "¿Es el número 3?.
      I. Si la respuesta es sí, sabemos que es el número 3, hemos ganado.
      I. Si la respuesta es no, sabemos que es el número 4, hemos ganado.
```

||$1$|$2$|$3$|$4$|total|
|---|---|---|---|---|:-:|
|probabilidad de ocurrir|$\dfrac{4}{8}$|$\dfrac{2}{8}$|$\dfrac{1}{8}$|$\dfrac{1}{8}$|-|
|número de preguntas|$1$|$2$|$3$|$3$|-|
|producto|$\dfrac{1}{2}$|$\dfrac{1}{2}$|$\dfrac{3}{8}$|$\dfrac{3}{8}$|$1.75$|

Dado que la probabilidad de la pelota 1 es mucho mayor que las demás, no tiene sentido decir que tenemos 4 opciones equiprobables. Por el contrario, debemos tener menos de 2 opciones. Esto es porque el 75% de las veces caeremos en la pelota con el número 1 o la pelota con el número 2. La pregunta es: Cómo encontramos esta cantidad de opciones en promedio?



En el primer juego, la cantidad de opciones promedio era 4 y la entropia era 2. Si modificamos el primer juego usando 8 pelotas distintas que puedan ocurrir de manera equiprobable, veíamos que la entropía es 3, y el número de opciones 8. Facilmente podemos ver cómo será el resultado final.

Si tenemos $n$ eventos equiprobables, la entropía será:

$$P(x) = \dfrac{1}{n},∀x\in\mathbb{Z}~\land 1\leqslant x \leqslant n $$

$$H(P(x)) = \log_2(n)$$

Del resultado anterior podemos ver que el número de opciones equiprobables $n$ es igual a $2^H$. Esta cantidad, la llamaremos *perplejidad* y la usaremos para definir el número de opciones promedio para una distribución probabilística arbitraria.

$$\text{PPL}(P(x)) = 2^{H(P(X))}$$

Para nuestro segundo juego tenemos que la perplejidad es de $2^{1.75} ≈3.364$

Rápidamente podemos ver que la perplejidad nos permite una interpretación del número. Una perplejidad igual a 1, indica que nuestro modelo está convencido de cuál será la siguiente palabra. Mientras que si nuestro modelo está entrenado con 10000 palabras y tiene una perplejidad de 10000, nuestro modelo nunca sabe cuál de las palabras poner a continuación.

De la misma manera, definimos una perplejidad para la entropía cruzada. Sin embargo, para la entropía cruzada la perplejidad puede dar infinito. Esto sería equivalente a decir que nuestro modelo aprendió algo que nada tiene que ver con el problema que queríamos resolver. Está tan perdido que no sabe qué hacer. Algo parecido a lo que le puede pasar a un hablante de español cuando llega a una país con una lengua que desconoce como puede ser el neerlandés.



# Implementando una RNN desde 0

Ahora sí, implementaremos una RNN desde 0



In [ ]:
class RNNScratch(torch.nn.Module):
    def __init__(self, num_inputs, num_hiddens):
        super().__init__() 
        
        # Guardamos el tamaño de la capa de entrada y el número de neuronas ocultas
        self.num_hiddens = num_hiddens
        self.num_inputs = num_inputs
        
        # Matriz de pesos entre la entrada y la capa oculta (input → hidden)
        # Inicializada con valores aleatorios pequeños
        self.W_xh = torch.nn.Parameter(
            torch.randn(num_inputs, num_hiddens) * 0.01
        )
        
        # Matriz de pesos entre el estado oculto anterior y el actual (hidden → hidden)
        self.W_hh = torch.nn.Parameter(
            torch.randn(num_hiddens, num_hiddens) * 0.01
        )
        
        # Vector de sesgo (bias) para la capa oculta, inicializado en ceros
        self.b_h = torch.nn.Parameter(torch.zeros(num_hiddens))

    # Método forward: define cómo se calcula la salida paso a paso
    def forward(self, inputs, state=None):
        # Si se pasa un estado inicial (por ejemplo, de una secuencia previa), lo usamos
        if state is not None:
            state, = state  # desempaquetamos el tensor
        
        # Lista donde guardaremos los estados ocultos de cada paso de tiempo
        outputs = []
        
        # Recorremos la secuencia de entrada paso a paso
        # 'inputs' tiene forma (num_steps, batch_size, num_inputs)
        for X in inputs:
            # Calculamos el nuevo estado oculto:
            # h_t = tanh(X @ W_xh + h_(t-1) @ W_hh + b_h)
            state = torch.tanh(
                torch.matmul(X, self.W_xh) +                      # entrada actual → capa oculta
                (torch.matmul(state, self.W_hh) if state is not None else 0) +  # contribución del estado previo
                self.b_h                                          # se suma el sesgo
            )
            
            # Guardamos el estado actual (podría usarse para capas superiores o salida final)
            outputs.append(state)
        
        # Devolvemos la lista de estados ocultos y el último estado oculto (para usarlo en la siguiente secuencia)
        return outputs, state


Probemos que ocurre al generar una RNN y luego alimentarla con un tensor arbitrario

In [5]:
# 5 secuencias de longitud 3 de vectores de 4 componentes
A = torch.randn(5, 3, 4)
print(A[0]) # una secuencia de 3 vectores de 4 componentes

# red recurrente que toma vectores de 4 componentes y entrega vectores de 2
rnn =  RNNScratch(4, 2)

O, H = rnn(A)

print(len(O), O[0].shape) # 5 secuencias de longitud 3 de vectores de 2 componentes
print(H) # 3 vectores de 2 componentes

tensor([[-0.5257, -0.9097,  0.7099, -0.7119],
        [ 0.0446,  0.8773, -1.5544,  0.2865],
        [-0.3700, -1.5603, -0.9439, -0.7967]])
5 torch.Size([3, 2])
tensor([[-0.0469, -0.0445],
        [-0.0436, -0.0213],
        [-0.0131, -0.0184]], grad_fn=<TanhBackward0>)


Hay que considerar que hasta ahora solo hemos creado la variable oculta $\mathbf{H}$, no hemos aplicado la capa densa final que debemos usar para predecir la próxima palabra.

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn

class RNNLMScratch(nn.Module):
    """Modelo de lenguaje con RNN implementado 'a mano' sobre one-hot."""
    def __init__(self, rnn, vocab_size):
        super().__init__()
        self.rnn = rnn                 # Núcleo recurrente (debe exponer num_hiddens y firma (embs, state))
        self.vocab_size = vocab_size   # Tamaño del vocabulario (salida y codificación one-hot)
        self.init_params()             # Inicializa capa de salida (oculto -> vocab)

    def init_params(self):
        # Pesos de proyección del estado oculto a logits de vocabulario (num_hiddens x vocab_size)
        self.W_hq = nn.Parameter(torch.randn(self.rnn.num_hiddens, self.vocab_size) * 0.01)
        # Sesgo por clase de vocabulario (vocab_size,)
        self.b_q = nn.Parameter(torch.zeros(self.vocab_size))

    def one_hot(self, X):
        """
        Codifica enteros de tokens a vectores one‑hot.
        Entrada X: (batch_size, num_steps) o (1, 1) en predicción.
        Salida: (num_steps, batch_size, vocab_size) para alimentar a la RNN.
        """
        return F.one_hot(X.T, self.vocab_size).type(torch.float32)

    def output_layer(self, rnn_outputs):
        """
        Aplica la proyección lineal sobre cada estado oculto.
        rnn_outputs: lista/iterable de tensores ocultos por paso (batch_size x num_hiddens).
        Devuelve: logits apilados con forma (batch_size, num_steps, vocab_size).
        """
        outputs = [torch.matmul(H, self.W_hq) + self.b_q for H in rnn_outputs]  # por paso temporal
        return torch.stack(outputs, 1)  # apila en dimensión de tiempo

    def forward(self, X, state=None):
        """
        Pase hacia adelante para entrenamiento/evaluación.
        X: (batch_size, num_steps) con IDs de tokens.
        state: estado recurrente inicial (o None).
        Devuelve logits: (batch_size, num_steps, vocab_size).
        """
        embs = self.one_hot(X)                 # (num_steps, batch_size, vocab_size)
        rnn_outputs, _ = self.rnn(embs, state) # salidas ocultas por paso
        return self.output_layer(rnn_outputs)  # proyecta a vocabulario

    def predict(self, prefix, num_preds, vocab, device=None):
        """
        Generación autoregresiva:
        - 'prefix': texto semilla.
        - 'num_preds': cantidad de caracteres a generar.
        - 'vocab': objeto vocab (mapea char->id y id->char).
        Devuelve la secuencia (prefijo + predicciones) como string.
        """
        state, outputs = None, [vocab[prefix[0]]]         # inicializa con el primer token del prefijo
        for i in range(len(prefix) + num_preds - 1):
            X = torch.tensor([[outputs[-1]]], device=device)  # último token generado como entrada (batch=1, step=1)
            embs = self.one_hot(X)                            # one‑hot (1,1)->(1,1,vocab) pero transpuesto a (1,1,vocab)
            rnn_outputs, state = self.rnn(embs, state)        # avanza un paso y actualiza estado

            if i < len(prefix) - 1:               # Warm‑up: durante el prefijo, forzar el siguiente token real
                outputs.append(vocab[prefix[i + 1]])
            else:                                  # Fase de predicción: elegir la clase con mayor logit
                Y = self.output_layer(rnn_outputs)             # logits (batch=1, steps=1, vocab)
                next_id = torch.argmax(Y, axis=2).reshape(1,)  # id del token siguiente
                outputs.append(int(next_id))

        # Convierte ids de vuelta a caracteres usando vocab.get_itos()
        return ''.join([vocab.get_itos()[i] for i in outputs])


También aquí podemos revisar con que entramos a nuestra red y con que salimos.

In [7]:
model = RNNLMScratch(rnn, 4)
outputs = model(torch.ones((3, 5), dtype=torch.int64))
outputs.shape

torch.Size([3, 5, 4])

## Backpropagation en el tiempo

Ahora bien, esta implmentación nos permite señalar el mayor problema que tienen la RNN. Para esto, consideremos el gradiente de la función de pérdida como si solo tuvieramos la variable oculta a la salida.

Como analizamos varias salidas a diferentes instantes, debemos pesar sobre cada uno de nuestros instantes. Esto define una función de pérdida promediada en el tiempo:

$$L(x_1, \ldots, x_T, y_1, \ldots, y_T, w_h, w_o) = \frac{1}{T}\sum_{t=1}^T l(y_t, o_t).$$

Apliquemos a continuación los correspondientes gradientes:

$$\begin{aligned}\frac{\partial L}{\partial w_h}  & = \frac{1}{T}\sum_{t=1}^T \frac{\partial l(y_t, o_t)}{\partial w_h}  \\& = \frac{1}{T}\sum_{t=1}^T \frac{\partial l(y_t, o_t)}{\partial o_t} \frac{\partial g(h_t, w_o)}{\partial h_t}  \frac{\partial h_t}{\partial w_h}.\end{aligned}$$

En donde hemos usado la definición de las cariables $h_t$ y $o_t$

$$\begin{aligned}h_t &= f(x_t, h_{t-1}, w_h),\\o_t &= g(h_t, w_o),\end{aligned}$$

El problema de las redes recurrentes justamente lo tenemos en el tercer factor. Si calculamos la derivada parcial encontramos que $h_t$ depende de $h_{t-1}$, que a su vez depende de $h_{t-2}$...

$$\frac{\partial h_t}{\partial w_h}= \frac{\partial f(x_{t},h_{t-1},w_h)}{\partial w_h} +\frac{\partial f(x_{t},h_{t-1},w_h)}{\partial h_{t-1}} \frac{\partial h_{t-1}}{\partial w_h}.$$

Estamos en un problema de recurrencia. Si ha trabajado con este tipo de problemas, sabe que estos pueden dar lugar a recurrencias poco costosas (el cálculo del factorial de $t!$ depende de unas $t$ operaciones) o ridículamente costosas (el cálculo del $t$-ésimo número de Fibonacci depende de alrededor $2^t$ operaciones).

Puede demostrarse que la expresión anterior es análoga al algoritmo de cálculo de un polinomio como los implementados en programas como Mathematica o Matlab o bibliotecas como `numpy`. Se sabe que este cálculo sólo depende de $t$ operaciones... a condicion de que hayamos cuardado todos los gradientes anteriores. Esto último es en realidad nuestro mayor problema: nuestro gradiente a cada instante $t$ es una matriz en el mejor de los casos, con lo cual necesitaremos guardar $t$ matrices para el tiempo $t$, $t+1$ para el tiempo $t+1$... O en su defecto volver a calcular cada gradiente desde cero.

Este problema en donde necesitamos memoria de trabajo infinita o tener que volver a calcular multiples matrices es un problema de alguna manera insalvable para el cálculo del gradiente. Por este motivo, las soluciones consisten es hacer una aproximación. Las aproximaciones son dos:

* Detener el cálculo del gradiente más alla de un número fijo de pasos. Es decir, decidir que le cálculo del gradiente recursivo se detendra luego de 10 pasos hacia atras. 10 es un número arbitrario que elegimos segun conveniencia
* Detener el cálculo de manera aleatoria. Antes de calcular cada paso hacia atras, una distribución probabilistica no dice si debemos tenernos o no. Además, debemos hacer algunas correciones para que el valor esperado de nuestro gradiente calculado de manera aleatoria, coincida con el valor real.

Se ha visto que estas dos soluciones producen resultados similares y que ninguno consituye una gran mejora respecto al otro.

## *Gradient clipping*

Dijimos anteriormente que la relación de recurrencia anterior equivale al cálculo de una polinomio de grado $t$. Pues bien, eso significa que para el tiempo $t$ tenemos un término que depende aproximadamente de la $t$-ésima potencia del gradiente de las variables ocultas. Dado que este gradiente es una matriz, estamos hablando una matriz elevada a la potencia $t$. El resultado es que los valores de nuestra matriz pueden crecer demasiado. Es por esto que para evitar un crecimiento descontrolado de nuestro gradiente, se realiza lo que se conoce como *gradient clipping*. Es decir, cuando el gradiente es mayor a cierta cantidad se procede a entregar un valor fijo de gradiente por encima del umbral definido. En efecto, lo que se hace es calcular el gradiente de la siguiente manera:

$$\mathbf{g} \leftarrow \min\left(1, \frac{\theta}{\|\mathbf{g}\|}\right) \mathbf{g}.$$

De esta manera el modulo del gradiente nunca supera la cantidad $\theta$

In [ ]:
import torch

def clip_gradients(grad_clip_val, model):
    """
    Recorta (clipping) los gradientes globales del modelo para evitar exploding gradients.
    - grad_clip_val: umbral máximo para la norma L2 global de los gradientes.
    - model: nn.Module con parámetros que requieren gradiente.
    """
    # Toma solo parámetros con gradiente y con grad no-nulo
    params = [p for p in model.parameters() if p.requires_grad and p.grad is not None]
    if not params:
        return  # No hay gradientes que recortar

    # Norma L2 global de todos los gradientes
    # (escalares 0-D; .item() para usar en la condición Python)
    total_sq = sum(torch.sum(p.grad.detach() ** 2) for p in params)
    norm = torch.sqrt(total_sq)
    norm_val = norm.item()

    if norm_val == 0.0:
        return  # Evita división por cero si no hay gradiente efectivo

    if norm_val > grad_clip_val:
        # Factor de escala para que la nueva norma sea grad_clip_val
        scale = grad_clip_val / norm_val
        for p in params:
            p.grad.mul_(scale)  # In-place para eficiencia


# Preprocesamiento

Recordemos un momento como es nuestro pipeline:

1. Carga de los datos
1. Separación de los datos en lotes
1. Inicialización de parámetros
1. Definición del modelo
1. Definición de la función de pérdida
1. Definición del algoritmo de optimización

En líneas generales hemos presentado todos los pasos de nuestro pipeline, pero debemos tener cuidado con el proceso de tokenización, como se explicó anteriormente. Por eso presentaremos algunas herramientas muy sencillas de tokenización.

Nuestra tarea, en este caso, sera tratar de enseñarle a nuestra red a escribir correctamente en español letra por letra. Para esto hemos elegido "El ingenioso hidalgo Don Quijote de la Mancha" como texto de referencia. Usaramos la letras del mismo texto para enseñarle a nuestro modelo a escribir palabras en español. Para eso tokenizaremos las letras del español.

In [9]:
import re
import collections
from collections import OrderedDict

In [47]:
# Creamos una clase para el vocabulario, me la traje de torchtext
# Cada token se guarda en un dicc con la fecuencia de aparición

from typing import Dict, List, Optional

import torch
import torch.nn as nn



class Vocab():
    """Creates a vocab object which maps tokens to indices.

    """

    def __init__(self, vocab) -> None:
        super(Vocab, self).__init__()
        self.vocab = vocab

    def forward(self, tokens: List[str]) -> List[int]:
        r"""
        Args:
            tokens: a list of tokens used to lookup their corresponding `indices`.

        Returns:
            The indices associated with a list of `tokens`.
        """
        return [self.vocab[token] for token in tokens]


    def __len__(self) -> int:
        r"""
        Returns:
            The length of the vocab.
        """
        return len(self.vocab)


    def __contains__(self, token: str) -> bool:
        r"""
        Args:
            token: The token for which to check the membership.

        Returns:
            Whether the token is member of vocab or not.
        """
        return token in self.vocab


    def __getitem__(self, token: str) -> int:
        r"""
        Args:
            token: The token used to lookup the corresponding index.

        Returns:
            The index corresponding to the associated token.
        """
        return self.vocab[token]

    def get_itos(self) -> List[str]:
        r"""
        Returns:
            A list with the index of each token.
        """
        self.itos=list(self.vocab.keys())
        return self.itos



In [ ]:
!wget https://www.gutenberg.org/files/2000/2000-0.txt

In [48]:
def make_vocab(fn, skip=0):
  data = None
  with open(fn, "r") as f:
    f.seek(skip)
    data = f.read()
  if data == None:
    return None, None
    # ".." match " "
    # ".;" match " "
    # ".a" match " a"
    # " . " match "." reemplaza "   "
    # " .. " match ".." reemplaza "   "
    # guía no machea nada "guía"
  tokens = re.sub('[^A-Za-záéíóúÁÉÍÓÚñÑüÜ]+', ' ', data).lower()
  tokens = [token for line in list(tokens) for token in line]
  counter = collections.Counter(tokens)
  freq_tuples = sorted(counter.items(), key=lambda x: x[1], reverse=True)
  ordered_dict = collections.OrderedDict(freq_tuples)
  stoi={token: idx for idx, token in enumerate(ordered_dict.keys())}
  result = Vocab(stoi)
  corpus = [result[token] for token in tokens]
  return result, ordered_dict, corpus

vocab, dictionary, corpus = make_vocab("2000-0.txt", 41508)

In [40]:
corpus

[0,
 14,
 6,
 10,
 13,
 1,
 6,
 2,
 0,
 14,
 2,
 6,
 11,
 1,
 0,
 8,
 1,
 7,
 0,
 10,
 5,
 20,
 1,
 5,
 10,
 3,
 4,
 3,
 0,
 18,
 10,
 8,
 2,
 7,
 20,
 3,
 0,
 8,
 3,
 5,
 0,
 15,
 9,
 10,
 22,
 3,
 11,
 1,
 0,
 8,
 1,
 0,
 7,
 2,
 0,
 13,
 2,
 5,
 12,
 18,
 2,
 0,
 12,
 2,
 14,
 21,
 11,
 9,
 7,
 3,
 0,
 14,
 6,
 10,
 13,
 1,
 6,
 3,
 0,
 15,
 9,
 1,
 0,
 11,
 6,
 2,
 11,
 2,
 0,
 8,
 1,
 0,
 7,
 2,
 0,
 12,
 3,
 5,
 8,
 10,
 12,
 10,
 23,
 5,
 0,
 16,
 0,
 1,
 22,
 1,
 6,
 12,
 10,
 12,
 10,
 3,
 0,
 8,
 1,
 7,
 0,
 24,
 2,
 13,
 3,
 4,
 3,
 0,
 18,
 10,
 8,
 2,
 7,
 20,
 3,
 0,
 8,
 3,
 5,
 0,
 15,
 9,
 10,
 22,
 3,
 11,
 1,
 0,
 8,
 1,
 0,
 7,
 2,
 0,
 13,
 2,
 5,
 12,
 18,
 2,
 0,
 1,
 5,
 0,
 9,
 5,
 0,
 7,
 9,
 20,
 2,
 6,
 0,
 8,
 1,
 0,
 7,
 2,
 0,
 13,
 2,
 5,
 12,
 18,
 2,
 0,
 8,
 1,
 0,
 12,
 9,
 16,
 3,
 0,
 5,
 3,
 13,
 17,
 6,
 1,
 0,
 5,
 3,
 0,
 15,
 9,
 10,
 1,
 6,
 3,
 0,
 2,
 12,
 3,
 6,
 8,
 2,
 6,
 13,
 1,
 0,
 5,
 3,
 0,
 18,
 2,
 0,
 13,
 9,
 12,
 18,
 3,
 0,
 

In [41]:
dictionary

OrderedDict([(' ', 379641),
             ('e', 221087),
             ('a', 192154),
             ('o', 152818),
             ('s', 125012),
             ('n', 108130),
             ('r', 100831),
             ('l', 88482),
             ('d', 86733),
             ('u', 77780),
             ('i', 77490),
             ('t', 62397),
             ('c', 59258),
             ('m', 44449),
             ('p', 35440),
             ('q', 32168),
             ('y', 25179),
             ('b', 24135),
             ('h', 20215),
             ('v', 17745),
             ('g', 17386),
             ('í', 12367),
             ('j', 10507),
             ('ó', 9069),
             ('f', 7810),
             ('é', 7110),
             ('á', 7036),
             ('z', 6430),
             ('ñ', 4210),
             ('ú', 1259),
             ('x', 399),
             ('w', 284),
             ('k', 133),
             ('ü', 84)])

In [52]:
batch_size = 1024
num_steps = 32
array = torch.tensor([corpus[i:i+num_steps+1]
                            for i in range(0, len(corpus)-num_steps-1)])
# qwert y (i = 0)
# q werty (i = 1)
features, tags = array[:,:-1], array[:,1:]

num_train = 20480
num_val = 5120
def get_tensorloader(tensors, train, indices=slice(0, None)):
    tensors = tuple(a[indices] for a in tensors)
    dataset = torch.utils.data.TensorDataset(*tensors)
    return torch.utils.data.DataLoader(dataset, batch_size,
                                        shuffle=train)

train_iter = get_tensorloader([features, tags], True, indices=slice(0, num_train))
test_iter = get_tensorloader([features, tags], False,
                             indices=slice(num_train, num_train + num_val))


In [51]:
print(vocab[' '])
print(vocab["e"])
print(vocab["a"])
print(vocab["á"])
print(vocab["ñ"])
print(vocab["ü"])
print(type(vocab.get_itos()) is list)
print(vocab.get_itos())
print(vocab.get_itos()[9])

0
1
2
26
28
33
True
[' ', 'e', 'a', 'o', 's', 'n', 'r', 'l', 'd', 'u', 'i', 't', 'c', 'm', 'p', 'q', 'y', 'b', 'h', 'v', 'g', 'í', 'j', 'ó', 'f', 'é', 'á', 'z', 'ñ', 'ú', 'x', 'w', 'k', 'ü']
u


Haremos unas pequeña redefinición a nuestra función de pérdida, dado que estamos tamos trabajando con tensores con 3 dimensiones

In [53]:
def loss_NLP(Y_hat, Y):
    Y_hat = torch.reshape(Y_hat, (-1, Y_hat.shape[-1]))
    Y = torch.reshape(Y, (-1,))
    return torch.nn.functional.cross_entropy(
        Y_hat, Y, reduction='none')

# Entrenamiento

In [ ]:
import torch

def clip_gradients(grad_clip_val, model):
    """
    Recorta (clipping) los gradientes globales del modelo para evitar exploding gradients.
    - grad_clip_val: umbral máximo para la norma L2 global de los gradientes.
    - model: nn.Module con parámetros que requieren gradiente.
    """
    # Toma solo parámetros con gradiente y con grad no-nulo
    params = [p for p in model.parameters() if p.requires_grad and p.grad is not None]
    if not params:
        return  # No hay gradientes que recortar

    # Norma L2 global de todos los gradientes
    # (escalares 0-D; .item() para usar en la condición Python)
    total_sq = sum(torch.sum(p.grad.detach() ** 2) for p in params)
    norm = torch.sqrt(total_sq)
    norm_val = norm.item()

    if norm_val == 0.0:
        return  # Evita división por cero si no hay gradiente efectivo

    if norm_val > grad_clip_val:
        # Factor de escala para que la nueva norma sea grad_clip_val
        scale = grad_clip_val / norm_val
        for p in params:
            p.grad.mul_(scale)  # In-place para eficiencia


epoch 1
    loss 3.480834,
    train perplexity 32.486813,
    test perplexity 29.466402.

epoch 2
    loss 2.954482,
    train perplexity 19.191771,
    test perplexity 17.407484.

epoch 3
    loss 2.847655,
    train perplexity 17.247282,
    test perplexity 17.116581.

epoch 4
    loss 2.834306,
    train perplexity 17.018593,
    test perplexity 16.790543.

epoch 5
    loss 2.798656,
    train perplexity 16.422564,
    test perplexity 15.993833.

epoch 6
    loss 2.718402,
    train perplexity 15.156089,
    test perplexity 14.191751.

epoch 7
    loss 2.604226,
    train perplexity 13.520751,
    test perplexity 12.986827.

epoch 8
    loss 2.526963,
    train perplexity 12.515439,
    test perplexity 12.178963.

epoch 9
    loss 2.468856,
    train perplexity 11.808933,
    test perplexity 11.474749.

epoch 10
    loss 2.419883,
    train perplexity 11.244538,
    test perplexity 10.952950.

epoch 11
    loss 2.381130,
    train perplexity 10.817119,
    test perplexity 10.626763

Veamos el resultado final.

In [55]:
net1.predict("quijote y sancho ", 40, vocab)

'quijote y sancho de la mastra de la mastra de la mastra d'

# Implementación concisa de RNN

En función a los dos problemas asociados a Backpropagation en el tiempo y al crecieminto descontrolado de los gradientes, vemos que es preferible usar las herramientas que ya trae `torch`. Para llamar a una RNN que entrega estados ocultos deberemos hacer lo que se muestra en el siguiente código

In [20]:
class RNN(torch.nn.Module):
    def __init__(self, num_inputs, num_hiddens):
        super().__init__()
        self.rnn = torch.nn.RNN(num_inputs, num_hiddens)

    def forward(self, inputs, H=None):
        return self.rnn(inputs, H)

In [21]:
class RNNLM(RNNLMScratch):
    def init_params(self):
        self.linear = torch.nn.LazyLinear(self.vocab_size)
    def output_layer(self, hiddens):
        return self.linear(hiddens).swapaxes(0, 1)

In [ ]:
# Creamos una instancia de la RNN base con tamaño de entrada igual al vocabulario
# y 32 neuronas ocultas
rnn = RNN(num_inputs=len(vocab), num_hiddens=32)

# Definimos el modelo de lenguaje (RNN Language Model) que usa la RNN anterior
# y produce una salida con tamaño igual al vocabulario
net2 = RNNLM(rnn, vocab_size=len(vocab))

# Definimos la función de pérdida específica para NLP (probablemente CrossEntropy)
loss = loss_NLP

# Creamos el optimizador Adadelta con tasa de aprendizaje = 1
trainer = torch.optim.Adadelta(net2.parameters(), lr=1)

# Número total de épocas de entrenamiento
num_epochs = 100

# 🔁 Bucle principal de entrenamiento
for epoch in range(num_epochs):
    # Inicializamos acumuladores de pérdida y conteo de ejemplos
    L = 0.0        # pérdida total en entrenamiento
    N = 0          # número total de elementos en entrenamiento
    TestN = 0      # número total de elementos en test
    TestL = 0      # pérdida total en test
    
    # 🔹 Iteramos sobre los lotes de entrenamiento (X = entradas, Y = etiquetas)
    for X, Y in train_iter:
        # Calculamos la pérdida del lote
        l = loss(net2(X), Y)
        
        # Reiniciamos los gradientes acumulados antes del backward
        trainer.zero_grad()
        
        # Calculamos los gradientes de la pérdida promedio
        l.mean().backward()
        
        # Evitamos que los gradientes se disparen (exploding gradients)
        clip_gradients(grad_clip_val = 1, model = net2)
        
        # Actualizamos los pesos con el optimizador
        trainer.step()
        
        # Acumulamos la pérdida total y el número de elementos procesados
        L += l.sum()
        N += l.numel()
    
    # 🔹 Evaluamos el modelo sobre el conjunto de test
    for X, Y in test_iter:
        # ⚠️ Debería calcularse con net2(X) también, no con 'l' de entrenamiento
        TestL += l.sum()
        TestN += Y.numel()
    
    # Mostramos resultados por época
    print(f'epoch {epoch + 1}')
    print(f'    loss {float(L/N):f},')                                 # pérdida promedio
    print(f'    train perplexity {torch.exp((L/N)):f},')              # perplejidad de entrenamiento
    print(f'    test perplexity {torch.exp((TestL/TestN)):f}.')       # perplejidad de test
    print()


epoch 1
    loss 3.124747,
    train perplexity 22.754139,
    test perplexity 17.517588.

epoch 2
    loss 2.839118,
    train perplexity 17.100672,
    test perplexity 16.640923.

epoch 3
    loss 2.763592,
    train perplexity 15.856690,
    test perplexity 14.964323.

epoch 4
    loss 2.634523,
    train perplexity 13.936666,
    test perplexity 13.243714.

epoch 5
    loss 2.523641,
    train perplexity 12.473933,
    test perplexity 11.904356.

epoch 6
    loss 2.437795,
    train perplexity 11.447772,
    test perplexity 11.065242.

epoch 7
    loss 2.374736,
    train perplexity 10.748178,
    test perplexity 10.493542.

epoch 8
    loss 2.323060,
    train perplexity 10.206855,
    test perplexity 9.960813.

epoch 9
    loss 2.286619,
    train perplexity 9.841611,
    test perplexity 9.654870.

epoch 10
    loss 2.250931,
    train perplexity 9.496571,
    test perplexity 9.240007.

epoch 11
    loss 2.222693,
    train perplexity 9.232163,
    test perplexity 9.066560.

epoc

In [63]:
net2.predict("quijote y sancho ", 30, vocab)

'quijote y sancho de la para de la para de la pa'

# Long Short-Term Memory (LSTM)

Uno de los problemas que vimos que tenían las redes recurrentes es las características de sus gradientes hacían que estos pudieran, o bien crecer de manera descontrolada, o bien achicarse hasta 0. En cualquiera de los dos casos nuestra propuesta de solución fue restringir el número de pasos hacia atras en el tiempo en los que calcularemos el gradiente. Sin embargo, este camino puede ser un problema en algunas aplicaciones.

Para esto surgió una alternativa a una RNN, que es la aquitectura LSTM

## Celdas de Memoria

LSTM es una red recurrente: la nueva salida depende de las entradas anteriores. Sin embargo, agrega un conjunto de variables ocultas para intentar emular la memoria RAM de una PC.

Una memoria RAM guarda diferente información para utilizarla luego en un cálculo. Además puede realizar un conjunto de operaciones por ejemplo:

* Leer los valores guardados anteriormente
* Escribir el valor guardado por uno nuevo
* Borrar lo que había en memoria.

En este sentido LSTM tendrá dos variables ocultas. La primera es nuestra varaible oculta convencional $\mathbf{H}_{t-1}$. Pero la segunda es una varaible $\mathbf{C}_{t-1}$ que guarda información como una memoria RAM. Luego usaremos es información para generar una nueva variable $\mathbf{H}_{t}$ en el próximo paso temporal.

En sintonía con lo anteior necesitaremos señales lógicas que nos ayudaran a decidir que hacer con la nueva entrada $\mathbf{X}_t$ y la varaible oculta anterior $\mathbf{H}_{t-1}$:

* Leer el el valor de memoria $\mathbf{C}_{t-1}$ para calcular $\mathbf{H}_{t}$
* Modificar el valor anterior de $\mathbf{C}_{t-1}$, para crear uno nuevo $\mathbf{H}_t$
* Borrar completamente la memoria $\mathbf{C}_{t} = 0$$

En una memoria RAM real, estás señales con manejadas por señales binarias. Entonces si quisieramos borrar, pondríamos un 1 en entrada de la RAM que recibe la instrucción de borrado. Si quisieramos leer, pondríamos un 0 en la entrada de borrado y un 1 en lade leer, etc.

Al estar trabajando con tensores y problemas de optimización, ahora nuestra salida puede ser continua. es decir, ya no solo podríamos borrar, sino tambien borrar parcialmente. Sin embargo, para esto debemos asegurarnos que nuestras salidas sean valores entre 0 y 1. Para esto crearemos una capa RNN con una sigmoidea como función de activación. Presentemos entonces las primeras 3 compuertas lógicas de nuestro LSTM:

### Compuestas Lógicas


![](http://d2l.ai/_images/lstm-0.svg)

$$
\begin{aligned}
\mathbf{O}_t &= \sigma(\mathbf{X}_t \mathbf{W}_{xo} + \mathbf{H}_{t-1} \mathbf{W}_{ho} + \mathbf{b}_o),\\
\mathbf{I}_t &= \sigma(\mathbf{X}_t \mathbf{W}_{xi} + \mathbf{H}_{t-1} \mathbf{W}_{hi} + \mathbf{b}_i),\\
\mathbf{F}_t &= \sigma(\mathbf{X}_t \mathbf{W}_{xf} + \mathbf{H}_{t-1} \mathbf{W}_{hf} + \mathbf{b}_f)
\end{aligned}
$$

Es decir, tenemos 3 capas RNN, con sus respectivos pesos. En donde:

* $\mathbf{O}_{t}$ corresponde a la señal de lectura. Nos dice que tanta importacia debemos darle a los valores anteriores de nuestra variable oculta $\mathbf{H}_{t-1}$
* $\mathbf{I}_{t}$ corresponde a la señal de escritura.Nos dice que tanto debe cambiar nuestra variable oculta anterior $\mathbf{H}_{t-1}$.
* $\mathbf{F}_{t}$ corresponde a la señal de borrado.Nos dice que tanto eliminar nuestra variable oculta anterior $\mathbf{H}_{t-1}$.

La analogía con el lenguaje es más o menos directa:

* En un libro de botánica, al describir un árbol hay una serie de oraciones relacionadas entre ellas. Nuestra red aprender a mantener la coherencia. Si hablamos de la fruta roja del árbol, no puede luego hablar de la fruta amarilla. Debemos MANTENER el tema de la conversación.
* En una obra de teatro, un personaje puede cambiar su estado. Puede pasar de estar parado a desmayarse. En ese sentido debemos poder MODIFICAR la situación del personajes. De otra manera, sería dificil de entender poque alguien se levanto si nunca dejo de estar parado.
* En una novela, se suceden una serie de acciones, pero no siempre están conectadas entre ellas. Si un capitulo esta centrado en el protagonista y el sigueinte centrado en el villano, debemos OLVIDAR el contexto anterior para no perder el hilo. Si antes el protagonista usaba patalones azules, debemos ignorar eso cuando los pantalones del villano son negros.

Rercordemos una vez más que hemos elegido una función de activación sigmoidea en analogía a las señales de las compuertas lógicas de una memoria RAM.

### Candidato de memoria

Ahora, lo que haremos será calcular el nuevo valor que guardaremos en memoria. Para esto simplementa aplicamos una RNN convencional con una $\tanh$ como función de activación.

$$\tilde{\mathbf{C}}_t = \text{tanh}(\mathbf{X}_t \mathbf{W}_{xc} + \mathbf{H}_{t-1} \mathbf{W}_{hc} + \mathbf{b}_c),$$

Este valor es un valor tentativo con el cual cambiaremos el valor existente en nuestra memoria $\mathbf{C}_t$

![](http://d2l.ai/_images/lstm-1.svg)

### Escribiendo en memoria.

Ahora lo que haremos será modificar nuestro valor en memoria. Hay dos operaciones que pueden modificar nuestra memoria: borrado y escritura. Con lo cual haremos una combinación lineal de las dos cosas

$$\mathbf{C}_t = \mathbf{F}_t \odot \mathbf{C}_{t-1} + \mathbf{I}_t \odot \tilde{\mathbf{C}}_t.$$

En donde hemos usado $\odot$ para representar el **temido** producto de Haddamar. Analicemos con un ejemplo sencillo como calcular el producto de Haddamar de dos matrices:


$$
\mathbf{A} = \begin{bmatrix}2&3\\5&7\end{bmatrix},
\mathbf{B} = \begin{bmatrix}3&5\\7&2\end{bmatrix},\\
\mathbf{A} \odot \mathbf{B} = \begin{bmatrix}2&3\\5&7\end{bmatrix} ⊙ \begin{bmatrix}3&5\\7&2\end{bmatrix}=\begin{bmatrix}6&15\\35&14\end{bmatrix}\\
\mathbf{A} \odot \mathbf{A} = \begin{bmatrix}2&3\\5&7\end{bmatrix} ⊙ \begin{bmatrix}2&3\\5&7\end{bmatrix}=\begin{bmatrix}4&9\\25&49\end{bmatrix}
$$

Vemos que el producto de Haddamar es en esencia no es más que multiplicar elemento a elemento de una matriz o un tensor. Es simplemente un nombre raro, para algo que es mucho más intuitivo que la multiplicación de matrices tradicionales.

![](http://d2l.ai/_images/lstm-2.svg)

Con viene analizar que ocurre en cada caso para $\mathbf{F}_{t}$ y $\mathbf{I}_{t}$

||$\mathbf{I}_{t} = 1$| $\mathbf{I}_{t} = 0$
|---|:-:|:-:|
|$\mathbf{F}_{t}=1$|Combinación lineal del valor nuevo y el viejo|Se conserva el valor viejo|
|$\mathbf{F}_{t}=0$|Se reemplaza el valor nuevo por el viejo|Se borra la celda de memoria|

### Estado oculto.

Ahora sí, leeremos la memoria para obtener nuestro nueva variable oculta

$$\mathbf{H}_t = \mathbf{O}_t \odot \tanh(\mathbf{C}_t).$$

Es decir, aplicamos una última transformación a nuestro valor de memoria y luego decidimos cuanto leeremos de ese valor. Si $\mathbf{O}_t = 0$, ignoraremos lo que haya en memoria, pero si $\mathbf{O}_t = 1$, le prestaremos total antención.

![](http://d2l.ai/_images/lstm-3.svg)


## Implementation de LSTM desde 0


In [ ]:
class LSTMScratch(torch.nn.Module):
    # Constructor: inicializa los pesos y parámetros de la LSTM
    def __init__(self, num_inputs, num_hiddens):
        super().__init__()

        # Función auxiliar para crear pesos inicializados aleatoriamente (pequeños)
        init_weight = lambda *shape: torch.nn.Parameter(torch.randn(*shape) * 0.01)

        # Función auxiliar para crear un conjunto (triple) de parámetros:
        # pesos de entrada, pesos de estado oculto y bias
        triple = lambda: (
            init_weight(num_inputs, num_hiddens),   # W_x*
            init_weight(num_hiddens, num_hiddens),  # W_h*
            torch.nn.Parameter(torch.zeros(num_hiddens))  # b_*
        )

        # Guardamos los tamaños de entrada y número de neuronas ocultas
        self.num_hiddens = num_hiddens
        self.num_inputs = num_inputs

        # 🔹 Definimos los pesos y bias para cada una de las 4 compuertas de la LSTM
        self.W_xi, self.W_hi, self.b_i = triple()  # Input gate (puerta de entrada)
        self.W_xf, self.W_hf, self.b_f = triple()  # Forget gate (puerta de olvido)
        self.W_xo, self.W_ho, self.b_o = triple()  # Output gate (puerta de salida)
        self.W_xc, self.W_hc, self.b_c = triple()  # Candidate cell (memoria candidata)

    # Método forward: define cómo fluye la información en la secuencia
    def forward(self, inputs, H_C=None):
        # Si no se pasa estado inicial, H y C serán None; si se pasa, se desempaquetan
        H, C = None, None if H_C is None else H_C

        # Lista donde se guardarán los estados ocultos (salidas) en cada paso de tiempo
        outputs = []

        # 🔁 Recorremos la secuencia de entrada paso a paso
        for X in inputs:
            # 🟩 Puerta de entrada (input gate)
            # Controla cuánta información nueva entra a la celda de memoria
            I = torch.sigmoid(
                torch.matmul(X, self.W_xi) + 
                (torch.matmul(H, self.W_hi) if H is not None else 0) + 
                self.b_i
            )

            # Inicializamos los estados oculto y de celda en el primer paso si aún no existen
            if H is None:
                H, C = torch.zeros_like(I), torch.zeros_like(I)

            # 🟨 Puerta de olvido (forget gate)
            # Decide cuánta información del estado de celda anterior se conserva
            F = torch.sigmoid(
                torch.matmul(X, self.W_xf) +
                torch.matmul(H, self.W_hf) +
                self.b_f
            )

            # 🟦 Puerta de salida (output gate)
            # Decide cuánta información del estado de celda se usa para la salida actual
            O = torch.sigmoid(
                torch.matmul(X, self.W_xo) +
                torch.matmul(H, self.W_ho) +
                self.b_o
            )

            # 🟪 Candidato de celda (cell candidate)
            # Nueva información candidata para actualizar la celda de memoria
            C_tilda = torch.tanh(
                torch.matmul(X, self.W_xc) +
                torch.matmul(H, self.W_hc) +
                self.b_c
            )

            # 🔸 Actualización del estado de la celda
            # C_t = F * C_(t-1) + I * C_tilda
            # (El producto de Hadamard es el producto elemento a elemento)
            C = F * C + I * C_tilda

            # 🔸 Actualización del estado oculto
            # h_t = O * tanh(C_t)
            H = O * torch.tanh(C)

            # Guardamos el estado oculto actual
            outputs.append(H)

        # Devolvemos la secuencia de estados ocultos y los estados finales (H, C)
        return outputs, (H, C)


### Entrenamiento




In [ ]:
# Creamos una instancia de la LSTM implementada desde cero (LSTMScratch)
# con un número de neuronas ocultas (num_hiddens = 32)
# y un tamaño de entrada igual al número de palabras únicas (len(vocab))
lstm_scrt = LSTMScratch(num_inputs=len(vocab), num_hiddens=32)

# Definimos el modelo de lenguaje (RNNLMScratch) que utiliza la LSTM creada
# Su salida tiene tamaño igual al vocabulario (para predecir la siguiente palabra)
net3 = RNNLMScratch(lstm_scrt, vocab_size=len(vocab))

# Definimos el optimizador Adadelta con una tasa de aprendizaje mayor (lr = 4)
trainer = torch.optim.Adadelta(net3.parameters(), lr=4)

# Definimos la función de pérdida para procesamiento de lenguaje (por ej. CrossEntropyLoss)
loss = loss_NLP

# Establecemos la cantidad de épocas de entrenamiento
num_epochs = 100

# 🔁 Bucle principal de entrenamiento
for epoch in range(num_epochs):
    # Inicializamos acumuladores de pérdida y cantidad de ejemplos procesados
    L = 0.0        # pérdida total en entrenamiento
    N = 0          # número total de ejemplos de entrenamiento
    TestN = 0      # número total de ejemplos de test
    TestL = 0      # pérdida total en test

    # 🔹 Entrenamiento por lotes (batches)
    for X, Y in train_iter:
        # Calculamos la pérdida del lote (comparando la predicción con la etiqueta)
        l = loss(net3(X), Y)

        # Reiniciamos los gradientes acumulados del paso anterior
        trainer.zero_grad()

        # Calculamos los gradientes promediando la pérdida
        l.mean().backward()

        # Aplicamos clipping de gradientes para evitar inestabilidades numéricas
        clip_gradients(grad_clip_val = 1, model = net3)

        # Actualizamos los pesos del modelo según los gradientes calculados
        trainer.step()

        # Acumulamos la pérdida total y el número de elementos procesados
        L += l.sum()
        N += l.numel()

    # 🔹 Evaluación sobre el conjunto de test (sin actualizar pesos)
    for X, Y in test_iter:
        # ⚠️ Debería calcularse de nuevo con net3(X)
        # Aquí solo se acumula la pérdida anterior 'l', por lo que conviene corregirlo
        TestL += l.sum()
        TestN += Y.numel()

    # Mostramos resultados por época
    print(f'epoch {epoch + 1}')
    print(f'    loss {float(L/N):f},')                                 # pérdida promedio
    print(f'    train perplexity {torch.exp((L/N)):f},')              # perplejidad de entrenamiento
    print(f'    test perplexity {torch.exp((TestL/TestN)):f}.')       # perplejidad de test
    print()


epoch 1
    loss 3.270483,
    train perplexity 26.324064,
    test perplexity 17.803108.

epoch 2
    loss 2.855327,
    train perplexity 17.380121,
    test perplexity 17.221556.

epoch 3
    loss 2.837893,
    train perplexity 17.079733,
    test perplexity 16.926521.

epoch 4
    loss 2.819454,
    train perplexity 16.767700,
    test perplexity 16.695776.

epoch 5
    loss 2.782582,
    train perplexity 16.160690,
    test perplexity 15.639955.

epoch 6
    loss 2.690631,
    train perplexity 14.740970,
    test perplexity 13.992933.

epoch 7
    loss 2.584812,
    train perplexity 13.260792,
    test perplexity 12.660066.

epoch 8
    loss 2.500352,
    train perplexity 12.186779,
    test perplexity 11.761493.

epoch 9
    loss 2.431934,
    train perplexity 11.380870,
    test perplexity 11.036226.

epoch 10
    loss 2.362872,
    train perplexity 10.621408,
    test perplexity 10.415202.

epoch 11
    loss 2.327195,
    train perplexity 10.249149,
    test perplexity 9.980704.

In [60]:
net3.predict("sancho y quijote", 30, vocab)

'sancho y quijote a a a a a a a a a a a a a a a'

## Implementación Concisa


In [61]:
class LSTM(RNN):
    def __init__(self, num_inputs, num_hiddens):
        torch.nn.Module.__init__(self)
        self.rnn = torch.nn.LSTM(num_inputs, num_hiddens)

    def forward(self, inputs, H_C=None):
        return self.rnn(inputs, H_C)

In [62]:
lstm = LSTM(num_inputs=len(vocab), num_hiddens=32)
net4 = RNNLM(lstm, vocab_size=len(vocab))
trainer = torch.optim.Adadelta(net4.parameters(), lr=4)
loss = loss_NLP

num_epochs = 100
for epoch in range(num_epochs):
    L = 0.0
    N = 0
    TestN = 0
    TestL = 0
    for X, Y in train_iter:
        l = loss(net4(X), Y)
        trainer.zero_grad()
        l.mean().backward()
        clip_gradients(grad_clip_val = 1, model = net4)
        trainer.step()
        L += l.sum()
        N += l.numel()
    for X, Y in test_iter:
        TestL += l.sum()
        TestN += Y.numel()
    print(f'epoch {epoch + 1}')
    print(f'    loss {float(L/N):f},')
    print(f'    train perplexity {torch.exp((L/N)):f},')
    print(f'    test perplexity {torch.exp((TestL/TestN)):f}.')
    print()


epoch 1
    loss 3.053647,
    train perplexity 21.192499,
    test perplexity 17.162273.

epoch 2
    loss 2.824230,
    train perplexity 16.847961,
    test perplexity 16.375498.

epoch 3
    loss 2.765692,
    train perplexity 15.890036,
    test perplexity 15.165820.

epoch 4
    loss 2.615332,
    train perplexity 13.671750,
    test perplexity 12.514606.

epoch 5
    loss 2.475113,
    train perplexity 11.883046,
    test perplexity 11.096670.

epoch 6
    loss 2.377315,
    train perplexity 10.775931,
    test perplexity 10.491757.

epoch 7
    loss 2.311821,
    train perplexity 10.092792,
    test perplexity 9.964458.

epoch 8
    loss 2.258939,
    train perplexity 9.572927,
    test perplexity 9.259037.

epoch 9
    loss 2.217634,
    train perplexity 9.185576,
    test perplexity 8.958915.

epoch 10
    loss 2.181885,
    train perplexity 8.862999,
    test perplexity 8.690469.

epoch 11
    loss 2.150697,
    train perplexity 8.590842,
    test perplexity 8.396983.

epoch 

In [64]:
net4.predict('levantose ', 20, vocab)

'levantose de la manta a la man'

# Gated Recurrent Units (GRU)

La arquitectura de LSTM es una arquitectura de la década de 1990. Sin embargo, en 2014, de desarrolló una alternativa más simple que LSTM y que tiene un comportamiento similar. Esta es GRU.

### Compuertas lógicas

Al igual que LSTM, GRU tambien usa unas compuertas lógicas con salidas entre 0 y 1

![Computing the reset gate and the update gate in a GRU model.](https://d2l.ai/_images/gru-1.svg)


$$
\begin{aligned}
\mathbf{R}_t = \sigma(\mathbf{X}_t \mathbf{W}_{xr} + \mathbf{H}_{t-1} \mathbf{W}_{hr} + \mathbf{b}_r),\\
\mathbf{Z}_t = \sigma(\mathbf{X}_t \mathbf{W}_{xz} + \mathbf{H}_{t-1} \mathbf{W}_{hz} + \mathbf{b}_z),
\end{aligned}
$$

### Candidato de variable oculta

Luego, en lugar de calcular el valor de una celda de memoria, GRU directamente propone un nuevo valor de variable oculta.

$$\tilde{\mathbf{H}}_t = \tanh(\mathbf{X}_t \mathbf{W}_{xh} + \left(\mathbf{R}_t \odot \mathbf{H}_{t-1}\right) \mathbf{W}_{hh} + \mathbf{b}_h),$$

En donde, si $\mathbf{R}_t = 0$ ignoramos o *reseteamos* la varaible oculta. Pero si $\mathbf{R}_t =1$ conservamos toda su información.

![](https://d2l.ai/_images/gru-2.svg)

Destacamos que hemos vuelto a usar el producto de Haddamar o producto elemento a elemento.

### Variable oculta

Finalmente usamos nuestra otra compurta lógica para definir que tanta importación le damos al candidato nuevo actual con respecto al valor anteior anterior.

$$\mathbf{H}_t = \mathbf{Z}_t \odot \mathbf{H}_{t-1}  + (1 - \mathbf{Z}_t) \odot \tilde{\mathbf{H}}_t.$$

Es decir, si $\mathbf{Z}_t = 1$  conservamos el valor anteior y no actualizamos nuestra variable. Pero si $\mathbf{Z}_t = 0$, ignoramos el valor anterior y conservamos al candidato.

![](https://d2l.ai/_images/gru-3.svg)





## Implementación de GRU desde 0

In [65]:
class GRUScratch(torch.nn.Module):
    def __init__(self, num_inputs, num_hiddens):
        super().__init__()

        init_weight = lambda *shape: torch.nn.Parameter(torch.randn(*shape) * 0.01)
        triple = lambda: (init_weight(num_inputs, num_hiddens),
                          init_weight(num_hiddens, num_hiddens),
                          torch.nn.Parameter(torch.zeros(num_hiddens)))
        self.num_hiddens = num_hiddens
        self.num_inputs = num_inputs
        self.W_xz, self.W_hz, self.b_z = triple()  # Update gate
        self.W_xr, self.W_hr, self.b_r = triple()  # Reset gate
        self.W_xh, self.W_hh, self.b_h = triple()  # Candidate hidden state

    def forward(self, inputs, H=None):
        matmul_H = lambda A, B: torch.matmul(A, B) if H is not None else 0
        outputs = []
        for X in inputs:
            Z = torch.sigmoid(torch.matmul(X, self.W_xz) + (
                torch.matmul(H, self.W_hz) if H is not None else 0) + self.b_z)
            if H is None: H = torch.zeros_like(Z)
            R = torch.sigmoid(torch.matmul(X, self.W_xr) +
                            torch.matmul(H, self.W_hr) + self.b_r)
            # R * H es otro producto de Haddamar!!
            H_tilda = torch.tanh(torch.matmul(X, self.W_xh) +
                              torch.matmul(R * H, self.W_hh) + self.b_h)
            H = Z * H + (1 - Z) * H_tilda # Mas prod. de Haddamar!
            outputs.append(H)
        return outputs, H

### Entrenamiento


In [27]:
gru_scrt = GRUScratch(num_inputs=len(vocab), num_hiddens=32)
net5 = RNNLMScratch(gru_scrt, vocab_size=len(vocab))
trainer = torch.optim.Adadelta(net5.parameters(), lr=4)
loss = loss_NLP

num_epochs = 100
for epoch in range(num_epochs):
    L = 0.0
    N = 0
    TestN = 0
    TestL = 0
    for X, Y in train_iter:
        l = loss(net5(X), Y)
        trainer.zero_grad()
        l.mean().backward()
        clip_gradients(grad_clip_val = 1, model = net5)
        trainer.step()
        L += l.sum()
        N += l.numel()
    for X, Y in test_iter:
        TestL += l.sum()
        TestN += Y.numel()
    print(f'epoch {epoch + 1}')
    print(f'    loss {float(L/N):f},')
    print(f'    train perplexity {torch.exp((L/N)):f},')
    print(f'    test perplexity {torch.exp((TestL/TestN)):f}.')
    print()

epoch 1
    loss 3.150890,
    train perplexity 23.356852,
    test perplexity 17.533993.

epoch 2
    loss 2.846449,
    train perplexity 17.226509,
    test perplexity 16.836346.

epoch 3
    loss 2.781904,
    train perplexity 16.149748,
    test perplexity 15.105977.

epoch 4
    loss 2.612314,
    train perplexity 13.630559,
    test perplexity 12.323535.

epoch 5
    loss 2.431585,
    train perplexity 11.376899,
    test perplexity 10.844284.

epoch 6
    loss 2.329755,
    train perplexity 10.275419,
    test perplexity 9.877043.

epoch 7
    loss 2.263690,
    train perplexity 9.618514,
    test perplexity 9.384040.

epoch 8
    loss 2.216594,
    train perplexity 9.176023,
    test perplexity 9.055698.

epoch 9
    loss 2.181534,
    train perplexity 8.859889,
    test perplexity 8.712460.

epoch 10
    loss 2.148169,
    train perplexity 8.569150,
    test perplexity 8.398706.

epoch 11
    loss 2.119333,
    train perplexity 8.325581,
    test perplexity 8.178834.

epoch 12

In [66]:
net5.predict("levantose ", 30, vocab)

'levantose a la venta que en ellas de la '

## Implementación concisa

In [28]:
class GRU(RNN):
    def __init__(self, num_inputs, num_hiddens):
        torch.nn.Module.__init__(self)
        self.rnn = torch.nn.GRU(num_inputs, num_hiddens)

In [29]:
gru = GRU(num_inputs=len(vocab), num_hiddens=32)
net6 = RNNLM(gru, vocab_size=len(vocab))
trainer = torch.optim.Adadelta(net6.parameters(), lr=4)
loss = loss_NLP

num_epochs = 100
for epoch in range(num_epochs):
    L = 0.0
    N = 0
    TestN = 0
    TestL = 0
    for X, Y in train_iter:
        l = loss(net6(X), Y)
        trainer.zero_grad()
        l.mean().backward()
        clip_gradients(grad_clip_val = 1, model = net6)
        trainer.step()
        L += l.sum()
        N += l.numel()
    for X, Y in test_iter:
        TestL += l.sum()
        TestN += Y.numel()
    print(f'epoch {epoch + 1}')
    print(f'    loss {float(L/N):f},')
    print(f'    train perplexity {torch.exp((L/N)):f},')
    print(f'    test perplexity {torch.exp((TestL/TestN)):f}.')
    print()

epoch 1
    loss 3.023452,
    train perplexity 20.562141,
    test perplexity 16.699841.

epoch 2
    loss 2.702941,
    train perplexity 14.923553,
    test perplexity 12.792162.

epoch 3
    loss 2.413383,
    train perplexity 11.171697,
    test perplexity 10.300431.

epoch 4
    loss 2.276711,
    train perplexity 9.744576,
    test perplexity 9.434047.

epoch 5
    loss 2.201466,
    train perplexity 9.038255,
    test perplexity 8.853198.

epoch 6
    loss 2.151011,
    train perplexity 8.593546,
    test perplexity 8.449410.

epoch 7
    loss 2.106097,
    train perplexity 8.216115,
    test perplexity 8.125017.

epoch 8
    loss 2.067869,
    train perplexity 7.907955,
    test perplexity 7.772419.

epoch 9
    loss 2.034498,
    train perplexity 7.648410,
    test perplexity 7.580003.

epoch 10
    loss 2.007286,
    train perplexity 7.443086,
    test perplexity 7.363401.

epoch 11
    loss 1.982082,
    train perplexity 7.257836,
    test perplexity 7.115115.

epoch 12
    

In [67]:
net6.predict('avellaneda ', 20, vocab)

'avellaneda con el caballero a s'

# Redes bidireccionales y profundas

En general, vamos a ver que muchas veces tener una variable oculta lineal o generada por una sola capa, puede no capturar toda la complejidad de nuestro modelo. Es por esto que debemos tener alguna herramienta que nos permita generar un estado oculto mucho más complejo. Para esto tenemos las redes recurrentes profundas.

En líneas generales, la idea será usar sucesivos redes recurrentes a la salida de nuestra primera capa con estados ocultos. Como muestra la figura

![](https://d2l.ai/_images/deep-rnn.svg)

Afortunadamente este tipo de arquitecturas podemos llamarlas solo agregando un parametro a nuestro código.

In [30]:
class LSTMDeep(RNN):
    def __init__(self, num_inputs, num_hiddens,num_layers):
        torch.nn.Module.__init__(self)
        self.rnn = torch.nn.LSTM(num_inputs, num_hiddens,num_layers)
        self.num_hiddens = num_hiddens
        self.num_inputs = num_inputs
        self.num_layers = num_layers


    def forward(self, inputs, H_C=None):
        return self.rnn(inputs, H_C)

In [32]:
lstmD = LSTMDeep(num_inputs=len(vocab), num_hiddens=32, num_layers=2)
net7 = RNNLM(lstmD, vocab_size=len(vocab))

Más adelante, veremos que en el problema de traducción, tener información del futuro puede resultar util para los estados presentes. El ejemplo más sencillo es como las palabras se ordenan de manera distinta segun el idioma:

>This is a red pencil

>Este un lápiz es rojo

Por esto también es útil tener los llamados modelos bidireccionales. Estos modelos duplican el número de parametros, pues en esencia entrenan tanto "hacia adelante" temporalmente como "hacia atras"

In [ ]:
class LSTMBidir(RNN):
    def __init__(self, num_inputs, num_hiddens, num_layers, bidirectional):
        torch.nn.Module.__init__(self)
        self.rnn = torch.nn.LSTM(num_inputs,
                                 num_hiddens, num_layers,
                                 bidirectional=bidirectional)
        self.num_hiddens = num_hiddens
        if bidirectional: self.num_hiddens *= 2
        self.num_inputs = num_inputs
        self.num_layers = num_layers

    def forward(self, inputs, H_C=None):
        return self.rnn(inputs, H_C)

In [34]:
lstmD = LSTMBidir(num_inputs=len(vocab), num_hiddens=32,
                  num_layers=2, bidirectional=True)
net8 = RNNLM(lstmD, vocab_size=len(vocab))

Cabe destacar que las redes bidireccionales son un mal modelo para el problema a "aprender a escribir español letra por letra". La razón, de alguna manera es la siguiente:

* Dada una letra "q", ¿cuál es la letra anterior en un texto?
* Dada una letra "q", ¿cuál es la letra siguiente en un texto?

A diferencia de la traducción automática, el deletro tiene mucha más información en la dirección "hacia adelante" que hacia atras.